In [1]:
import pandas as pd

In [2]:
llm_error = pd.read_csv(
    "../llmerr.tsv", sep="\t", header=None, names=["model", "input", "result"]
)

FileNotFoundError: [Errno 2] No such file or directory: '../llmerr.tsv'

In [ ]:
llm_error

In [ ]:
llm_error.groupby(["model", "input", "result"]).size()

In [ ]:
llm_error

In [ ]:
llm_error_count = pd.DataFrame(llm_error.value_counts())

In [ ]:
llm_error_count = llm_error_count.reset_index()

In [ ]:
llm_error_count

In [ ]:
llm_error_count["total"] = llm_error_count.groupby("model")["count"].transform("sum")

In [ ]:
llm_error_count

In [ ]:
llm_error_count.sort_values(by=["model", "input", "result"], inplace=True)

In [ ]:
llm_error_count

In [ ]:
llm_error_count = llm_error_count.loc[:, ["total", "count", "model", "input", "result"]]

In [ ]:
llm_error_count = llm_error_count.sort_values(["total", "count"], ascending=False)

In [ ]:
llm_error_count.to_csv("output.tsv", sep="\t", index=False)

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np


def visualize_simulation_times(json_file):
    with open(json_file) as f:
        data = json.load(f)

    before_values = []
    after_values = []

    for entry in data["normalisation_output"]:
        outputs = entry["output"]
        seen_in_entry = []
        for result in outputs:
            value = result["value"]
            unit = result["unit"]

            if value is None:
                continue
            if unit == "ps":
                value = value / 1000
            elif unit == "μs":
                value = value * 1000
            elif unit == "s":
                value = value * 1000000000

            before_values.append(value)

            if value not in seen_in_entry:
                seen_in_entry.append(value)
                after_values.append(value)

    all_values = before_values + after_values
    min_val = min(all_values)
    max_val = max(all_values)
    bins = np.logspace(np.log10(min_val), np.log10(max_val), 30)

    plt.figure(figsize=(14, 5))
    plt.hist(before_values, bins=bins, alpha=0.8, label="Before Normalisation")
    plt.hist(after_values, bins=bins, alpha=0.8, label="After Normalisation")

    plt.xscale("log")
    plt.title("Simulation Time Normalisation Effect")
    plt.xlabel("Simulation Time (ns, log scale)")
    plt.ylabel("Number of entries")
    plt.legend()
    plt.tight_layout()
    plt.savefig("results/simulation_time_distribution.png")

In [ ]:
visualize_simulation_times(
    "results/norm_simu_times/normalized_simulation_time_deepseek.json"
)